In [17]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root added to path:", project_root)

Project root added to path: C:\Users\saw\Documents\fyp


In [18]:
import numpy as np
import pandas as pd
from pathlib import Path
# from sklearn.metrics import mean_squared_error

# Project imports
from src.preprocessing import get_preprocessor
from src.crossvalidation import get_cv
from src.models import get_rf_pipeline

In [19]:
DATA_PATH = Path("../data/raw/DatasetWithModules_Training.xlsx")
df = pd.read_excel(DATA_PATH)

def assign_perf_group(gpa):
    if gpa <= 2.99:
        return "Underperforming"
    elif gpa <= 3.29:
        return "Average"
    else:
        return "Performing"

df["GroupLabel"] = df["FinalGPA"].apply(assign_perf_group)

In [20]:
grade_map = {
    'A+': 11, 'A': 10, 'A-': 9, 'B+': 8, 'B': 7, 
    'B-': 6, 'C+': 5, 'C': 4, 'C-': 3, 'S': 2, 'F': 1
}

module_columns = ['Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'] # List your Semester X modules here

for col in module_columns:
    # We use .strip() in case there are hidden spaces like "A "
    df[col] = df[col].astype(str).str.strip().map(grade_map)
    df[col] = df[col].fillna(0)

In [22]:
numeric_features = [
    'Zscore', 'EnglishMarks',
]
numeric_features_sem1 = [
    'Zscore', 'EnglishMarks','S1'
]
numeric_features_sem2 = [
    'Zscore', 'EnglishMarks','S1', 'S2','Maths 2', 'MgtAccounting',
]
numeric_features_sem3 = [
    'Zscore', 'EnglishMarks','S1', 'S2', 'S3','Maths 2', 'MgtAccounting', 'StatsII', 'MIS'
]
numeric_features_sem4 = [
    'Zscore', 'EnglishMarks', 'S1', 'S2', 'S3', 'S4','Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'
]
numeric_features_sem5 = [
    'Zscore', 'EnglishMarks', 'S1', 'S2', 'S3', 'S4', 'S5', 'Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'
]
numeric_features_sem6 = [
    'Zscore', 'EnglishMarks', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'
]
numeric_features_sem7 = [
    'Zscore', 'EnglishMarks', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'
]
numeric_features_sem8 = [
    'Zscore', 'EnglishMarks',
    'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'Maths 2', 'MgtAccounting', 'StatsII', 'MIS', 'DataV'
]
categorical_features = [
    'Gender', 'Department', 'Province', 'MediumAL'
]

y = df["FinalGPA"]
strata = df["GroupLabel"]

In [ ]:
SEMESTER 0

In [23]:
import sklearn.metrics

X = df[numeric_features + categorical_features]

preprocessor = get_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

rf_model = get_rf_pipeline(
    preprocessor=preprocessor,
    max_depth=3,
    n_estimators=100,
)

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_list = []
overall_r2_list = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_val)

    rmse_sem0 = sklearn.metrics.root_mean_squared_error(y_val, y_pred)
    r2_sem0 = sklearn.metrics.r2_score(y_val, y_pred)
    
    overall_rmse_list.append(rmse_sem0)
    overall_r2_list.append(r2_sem0)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred[mask]
            
            rf_results[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results[g]['bias'].append(np.mean(actual - pred))

In [24]:
print(f"Mean RMSE: {np.mean(overall_rmse_list):.4f} (SD: {np.std(overall_rmse_list):.4f})")
print(f"Mean R2:   {np.mean(overall_r2_list):.4f} (SD: {np.std(overall_r2_list):.4f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s0 = np.array(rf_results[g]['rmse'])
    g_r2_s0 = np.array(rf_results[g]['r2'])
    g_bias_s0 = np.array(rf_results[g]['bias'])
    
    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s0):.4f} (SD: {np.std(g_rmse_s0):.4f})")
    print(f"   R2:   {np.mean(g_r2_s0):.4f} (SD: {np.std(g_r2_s0):.4f})")
    print(f"   Bias: {np.mean(g_bias_s0):.4f} (SD: {np.std(g_bias_s0):.4f})")

Mean RMSE: 0.3877 (SD: 0.0506)
Mean R2:   0.2950 (SD: 0.1095)

--- GROUP-WISE PERFORMANCE ---

>> Group: Performing
   RMSE: 0.3275 (SD: 0.0319)
   R2:   -1.8325 (SD: 0.8488)
   Bias: 0.2460 (SD: 0.0406)

>> Group: Underperforming
   RMSE: 0.5649 (SD: 0.1335)
   R2:   -3.2553 (SD: 3.5007)
   Bias: -0.4037 (SD: 0.1238)

>> Group: Average
   RMSE: 0.2033 (SD: 0.0517)
   R2:   -5.7687 (SD: 3.7573)
   Bias: -0.0888 (SD: 0.0442)


SEMESTER 1

In [25]:
preprocessor_s1 = get_preprocessor(
    numeric_features=numeric_features_sem1,
    categorical_features=categorical_features
)

rf_model_1 = get_rf_pipeline(
    preprocessor=preprocessor_s1,
    max_depth=3,
    n_estimators=100,
)

X = df[numeric_features_sem1 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_1 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_1 = []
overall_r2_1 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_1.fit(X_train, y_train)
    y_pred_1 = rf_model_1.predict(X_val)

    rmse_sem1 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_1)
    r2_sem1 = sklearn.metrics.r2_score(y_val, y_pred_1)

    overall_rmse_1.append(rmse_sem1)
    overall_r2_1.append(r2_sem1)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_1[mask]

            rf_results_1[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_1[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_1[g]['bias'].append(np.mean(actual - pred))

c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils

In [26]:
print("\n--- OVERALL PERFORMANCE (All Students) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_1):.4f} (SD: {np.std(overall_rmse_1):.4f})")
print(f"Mean R2:   {np.mean(overall_r2_1):.4f} (SD: {np.std(overall_r2_1):.4f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s1 = np.array(rf_results_1[g]['rmse'])
    g_r2_s1 = np.array(rf_results_1[g]['r2'])
    g_bias_s1 = np.array(rf_results_1[g]['bias'])
    
    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s1):.4f} (SD: {np.std(g_rmse_s1):.4f})")
    print(f"   R2:   {np.mean(g_r2_s1):.4f} (SD: {np.std(g_r2_s1):.4f})")
    print(f"   Bias: {np.mean(g_bias_s1):.4f} (SD: {np.std(g_bias_s1):.4f})")


--- OVERALL PERFORMANCE (All Students) ---
Mean RMSE: 0.2803 (SD: 0.0390)
Mean R2:   0.6267 (SD: 0.0857)

--- GROUP-WISE PERFORMANCE ---

>> Group: Performing
   RMSE: 0.2075 (SD: 0.0247)
   R2:   -0.1392 (SD: 0.3939)
   Bias: 0.1127 (SD: 0.0338)

>> Group: Underperforming
   RMSE: 0.4106 (SD: 0.1032)
   R2:   -1.3528 (SD: 1.8412)
   Bias: -0.1690 (SD: 0.1062)

>> Group: Average
   RMSE: 0.2124 (SD: 0.0617)
   R2:   -6.5726 (SD: 4.6913)
   Bias: -0.0488 (SD: 0.0579)


In [27]:
preprocessor_s2 = get_preprocessor(
    numeric_features=numeric_features_sem2,
    categorical_features=categorical_features
)

rf_model_2 = get_rf_pipeline(
    preprocessor=preprocessor_s2,
    max_depth=5,
    n_estimators=100,
)

X = df[numeric_features_sem2 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_2 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_2 = []
overall_r2_2 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_2.fit(X_train, y_train)
    y_pred_2 = rf_model_2.predict(X_val)

    rmse_sem2 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_2)
    r2_sem2 = sklearn.metrics.r2_score(y_val, y_pred_2)

    overall_rmse_2.append(rmse_sem2)
    overall_r2_2.append(r2_sem2)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_2[mask]

            rf_results_2[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_2[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_2[g]['bias'].append(np.mean(actual - pred))

c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils

In [28]:
print("\n--- OVERALL PERFORMANCE (All Students) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_2):.4f} (SD: {np.std(overall_rmse_2):.4f})")
print(f"Mean R2:   {np.mean(overall_r2_2):.4f} (SD: {np.std(overall_r2_2):.4f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s2 = np.array(rf_results_2[g]['rmse'])
    g_r2_s2 = np.array(rf_results_2[g]['r2'])
    g_bias_s2 = np.array(rf_results_2[g]['bias'])

    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s2):.4f} (SD: {np.std(g_rmse_s2):.4f})")
    print(f"   R2:   {np.mean(g_r2_s2):.4f} (SD: {np.std(g_r2_s2):.4f})")
    print(f"   Bias: {np.mean(g_bias_s2):.4f} (SD: {np.std(g_bias_s2):.4f})")


--- OVERALL PERFORMANCE (All Students) ---
Mean RMSE: 0.2508 (SD: 0.0362)
Mean R2:   0.7004 (SD: 0.0764)

--- GROUP-WISE PERFORMANCE ---

>> Group: Performing
   RMSE: 0.1754 (SD: 0.0200)
   R2:   0.1853 (SD: 0.2654)
   Bias: 0.0916 (SD: 0.0285)

>> Group: Underperforming
   RMSE: 0.3760 (SD: 0.0911)
   R2:   -0.9997 (SD: 1.4454)
   Bias: -0.1526 (SD: 0.1046)

>> Group: Average
   RMSE: 0.1971 (SD: 0.0475)
   R2:   -5.4287 (SD: 3.6596)
   Bias: -0.0427 (SD: 0.0581)


In [29]:
preprocessor_s3 = get_preprocessor(
    numeric_features=numeric_features_sem3,
    categorical_features=categorical_features
)

rf_model_3 = get_rf_pipeline(
    preprocessor=preprocessor_s3,
    max_depth=3,
    n_estimators=100,
)

X = df[numeric_features_sem3 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_3 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_3 = []
overall_r2_3 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_3.fit(X_train, y_train)
    y_pred_3 = rf_model_3.predict(X_val)

    rmse_sem3 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_3)
    r2_sem3 = sklearn.metrics.r2_score(y_val, y_pred_3)

    overall_rmse_3.append(rmse_sem3)
    overall_r2_3.append(r2_sem3)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_3[mask]

            rf_results_3[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_3[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_3[g]['bias'].append(np.mean(actual - pred))

c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\saw\Documents\fyp\venv_fyp\Lib\site-packages\sklearn\utils

In [30]:
print("\n--- OVERALL PERFORMANCE (SEM3) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_3):.4f} (SD: {np.std(overall_rmse_3):.4f})")
print(f"Mean R2:   {np.mean(overall_r2_3):.4f} (SD: {np.std(overall_r2_3):.4f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s3 = np.array(rf_results_3[g]['rmse'])
    g_r2_s3 = np.array(rf_results_3[g]['r2'])
    g_bias_s3 = np.array(rf_results_3[g]['bias'])

    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s3):.4f} (SD: {np.std(g_rmse_s3):.4f})")
    print(f"   R2:   {np.mean(g_r2_s3):.4f} (SD: {np.std(g_r2_s3):.4f})")
    print(f"   Bias: {np.mean(g_bias_s3):.4f} (SD: {np.std(g_bias_s3):.4f})")


--- OVERALL PERFORMANCE (SEM3) ---
Mean RMSE: 0.1986 (SD: 0.0330)
Mean R2:   0.8129 (SD: 0.0493)

--- GROUP-WISE PERFORMANCE ---

>> Group: Performing
   RMSE: 0.1400 (SD: 0.0197)
   R2:   0.4828 (SD: 0.1742)
   Bias: 0.0532 (SD: 0.0271)

>> Group: Underperforming
   RMSE: 0.3080 (SD: 0.0784)
   R2:   -0.4232 (SD: 1.7945)
   Bias: -0.1145 (SD: 0.0930)

>> Group: Average
   RMSE: 0.1312 (SD: 0.0260)
   R2:   -1.7917 (SD: 1.4874)
   Bias: -0.0160 (SD: 0.0403)
